# Notebook 07 — Classifier Selection

## Purpose
This notebook performs **classifier selection** to identify the most effective  
scikit-learn classification algorithms for distinguishing **AI-generated** and  
**real** images using the previously extracted **25-dimensional Digital Image  
Processing (DIP) feature vectors**.

Multiple classifiers are evaluated using the same normalized training data so  
that performance can be compared fairly and consistently. Rather than selecting  
a single model, the **top two performing classifiers** are identified and  
carried forward for subsequent training and final independent test evaluation.

---

## Inputs
This notebook uses the normalized training dataset generated by:

- `06_Normalize_and_Prepare_Classifier_Input.ipynb`

Expected input file:

- `train_feature_vectors_normalized.csv`

This dataset contains:

- Metadata columns:
  - `filename`
  - `class_label`
  - `source_dataset`
  - `subset`
- **25 normalized DIP feature columns**

---

## Local Execution Assumptions
This notebook is designed to run within the **local GitHub project structure**  
or a compatible environment (e.g., Google Colab).

It assumes:

- the project repository is available locally or cloned at runtime
- `src/project_config.py` is accessible
- prior pipeline notebooks have generated the required normalized dataset
- required Python packages (NumPy, pandas, scikit-learn) are installed

No external data downloads or preprocessing occur in this notebook.

---

## Important Note
All features have already been normalized using a scaler fit only on the  
training dataset in the previous notebook. No additional normalization  
should be applied here.

The independent test dataset exists, but it is **not used in this notebook**  
and remains reserved for final evaluation.

---

## Process Overview
This notebook evaluates multiple classifier types on the same normalized  
DIP feature set using a consistent cross-validation framework. The workflow includes:

1. loading and validating the normalized training dataset  
2. separating metadata from feature columns  
3. defining candidate classifiers  
4. evaluating baseline performance using stratified k-fold cross-validation  
5. ranking classifiers using multiple performance metrics  
6. applying **controlled (small-grid) hyperparameter tuning** to top-performing models  
7. saving comparison results for reporting and reproducibility  
8. selecting the **top two classifiers** for downstream training and evaluation  

---

## Outputs
This notebook produces:

- a ranked classifier comparison table for baseline models (including performance metrics and execution time)
- a ranked comparison of tuned classifier results
- saved CSV files:
  - `classifier_comparison_baseline.csv`
  - `classifier_comparison_tuned.csv`
- a JSON file containing the **top two classifier configurations**
- a summary identifying the recommended classifiers for subsequent notebooks

These outputs support both experimental reporting and downstream model training.

---

## Key Design Choice
This notebook performs **multi-model selection**, not single-model selection.

Rather than committing to a single classifier, the **two best-performing models**  
are selected based on cross-validated ROC-AUC after light hyperparameter tuning.

This design:

- reduces sensitivity to small cross-validation differences  
- enables comparison between different model families (e.g., kernel-based vs neural network)  
- strengthens experimental validity in the final evaluation stage  

---

## Classifiers Evaluated
The following classifier types are evaluated:

- Logistic Regression  
- Support Vector Machine (Linear and RBF kernels)  
- Random Forest  
- Extra Trees  
- Gradient Boosting  
- k-Nearest Neighbors (KNN)  
- Gaussian Naive Bayes  
- Multi-Layer Perceptron (MLP)  

---

## Scope Limitation
This notebook does **not** perform:

- exhaustive hyperparameter optimization  
- final model training on full datasets  
- evaluation on the independent test set  

These steps are performed in subsequent notebooks.

---

## Cell-by-Cell Structure

### Cell 1
Import required libraries.

### Cell 2
Load the normalized training dataset and preview its contents.

### Cell 3
Validate dataset integrity (missing values, class balance, feature count).

### Cell 4
Prepare feature matrix (`X_train`) and encoded target vector (`y_train`).

### Cell 5
Define candidate classifiers and cross-validation strategy.

### Cell 6
Run baseline classifier comparison using stratified k-fold cross-validation.

### Cell 7
Compile and rank baseline classifier results.

### Cell 8
Apply controlled hyperparameter tuning to top-performing classifiers.

### Cell 9
Save baseline and tuned results, and store the **top two classifier configurations**.

### Cell 10
Summarize results and recommend classifiers for downstream training and evaluation.

In [2]:
# ============================================
# Startup (Environment + Verification)
# ============================================

import os
import sys
from pathlib import Path

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_NORMALIZED_FILENAME,
    NUM_FEATURES,
    RANDOM_SEED,
)

# -------------------------------------------------
# Notebook 07 path overrides (minimal)
# -------------------------------------------------
METADATA_ROOT = REPO_DIR / "metadata"
INPUT_CSV_DIR = METADATA_ROOT / "vectors"
OUTPUT_DIR = REPO_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Define required file paths
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = INPUT_CSV_DIR / TRAIN_NORMALIZED_FILENAME

BASELINE_RESULTS_CSV = OUTPUT_DIR / "classifier_comparison_baseline.csv"
TUNED_RESULTS_CSV = OUTPUT_DIR / "classifier_comparison_tuned.csv"
BEST_CLASSIFIER_JSON = OUTPUT_DIR / "best_classifier_config.json"

# -------------------------------------------------
# Verify required input file
# -------------------------------------------------
print("Verifying required normalized training CSV file...\n")

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    raise FileNotFoundError(
        f"Missing required file in metadata/vectors/: "
        f"{TRAIN_NORMALIZED_FILENAME}"
    )

print("Required normalized training CSV file is present.")
print(f"Input file:         {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
print(f"Output directory:   {OUTPUT_DIR}")



Cloning repository...
Verifying required normalized training CSV file...

Required normalized training CSV file is present.
Input file:         /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv
Output directory:   /content/dip-ai-image-detection/models


In [3]:
# ============================================
# Cell 1: Import Required Libraries
# ============================================

import os
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# -------------------------------------------------
# Suppress warnings for cleaner notebook output
# -------------------------------------------------
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")



Libraries imported successfully.


In [4]:
# ============================================
# Cell 2: Load Normalized Training Data
# ============================================

print(f"Loading training data from: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")

# -------------------------------------------------
# Load dataset
# -------------------------------------------------
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

print("\nTraining dataset loaded successfully.")
print(f"Shape: {df_train.shape}")

# -------------------------------------------------
# Preview data
# -------------------------------------------------
print("\nFirst 5 rows:")
try:
    display(df_train.head())  # Works in Jupyter/Colab
except:
    print(df_train.head())    # Fallback for pure Python environments

# -------------------------------------------------
# Show column names
# -------------------------------------------------
print("\nColumn names:")
for col in df_train.columns:
    print(col)



Loading training data from: /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv

Training dataset loaded successfully.
Shape: (14400, 29)

First 5 rows:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939



Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std


In [5]:
# ============================================
# Cell 3: Validate Training Data
# ============================================

print("Performing sanity checks...\n")

# --------------------------------------------
# Check for missing values
# --------------------------------------------
missing_values = df_train.isnull().sum().sum()
print(f"Total missing values: {missing_values}")

if missing_values == 0:
    print("No missing values detected.")
else:
    print("WARNING: Missing values detected.")

# --------------------------------------------
# Check class distribution
# --------------------------------------------
print("\nClass distribution:")
class_counts = df_train["class_label"].value_counts()
print(class_counts)

# --------------------------------------------
# Verify metadata columns
# --------------------------------------------
expected_metadata_cols = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

print("\nChecking metadata columns...")
for col in expected_metadata_cols:
    if col in df_train.columns:
        print(f"OK: {col}")
    else:
        raise ValueError(f"Missing required column: {col}")

# --------------------------------------------
# Identify feature columns
# --------------------------------------------
feature_cols = [
    col for col in df_train.columns
    if col not in expected_metadata_cols
]

print(f"\nNumber of feature columns: {len(feature_cols)}")

# --------------------------------------------
# Verify feature count
# --------------------------------------------
EXPECTED_FEATURE_COUNT = NUM_FEATURES  # from project_config

if len(feature_cols) == EXPECTED_FEATURE_COUNT:
    print("Feature count is correct.")
else:
    raise ValueError(
        f"Expected {EXPECTED_FEATURE_COUNT} features, "
        f"found {len(feature_cols)}."
    )

print("\nValidation complete.")



Performing sanity checks...

Total missing values: 0
No missing values detected.

Class distribution:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Checking metadata columns...
OK: filename
OK: class_label
OK: source_dataset
OK: subset

Number of feature columns: 25
Feature count is correct.

Validation complete.


In [6]:
# ============================================
# Cell 4: Prepare X_train and y_train
# ============================================

print("Preparing feature matrix X and target vector y...\n")

# -------------------------------------------------
# Build X and y
# -------------------------------------------------
X_train = df_train[feature_cols].values
y_train = df_train["class_label"].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# -------------------------------------------------
# Encode labels
# -------------------------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

print("\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nEncoded class distribution:")
unique, counts = np.unique(y_train_encoded, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

print("\nPreparation complete.")



Preparing feature matrix X and target vector y...

X_train shape: (14400, 25)
y_train shape: (14400,)

Label encoding mapping:
ai -> 0
rl -> 1

Encoded class distribution:
Class 0: 7200
Class 1: 7200

Preparation complete.


In [7]:
# ============================================
# Cell 5: Define Candidate Classifiers
# ============================================

print("Defining candidate classifiers and scoring metrics...\n")

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

# -------------------------------------------------
# Define candidate classifiers
# -------------------------------------------------
candidate_classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    ),

    "Linear SVM": SVC(
        kernel="linear",
        probability=True,
        random_state=RANDOM_SEED
    ),

    "RBF SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_SEED
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Gaussian NB": GaussianNB(),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=RANDOM_SEED
    )
}

# -------------------------------------------------
# Define scoring metrics
# -------------------------------------------------
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print("Candidate classifiers:")
for name in candidate_classifiers.keys():
    print(f" - {name}")

print("\nScoring metrics:")
for metric in scoring_metrics.keys():
    print(f" - {metric}")

print("\nCross-validation strategy:")
print(" - StratifiedKFold")
print(" - n_splits = 5")
print(" - shuffle = True")
print(f" - random_state = {RANDOM_SEED}")

print("\nClassifier setup complete.")



Defining candidate classifiers and scoring metrics...

Candidate classifiers:
 - Logistic Regression
 - Linear SVM
 - RBF SVM
 - Random Forest
 - Extra Trees
 - Gradient Boosting
 - KNN
 - Gaussian NB
 - MLP

Scoring metrics:
 - accuracy
 - precision
 - recall
 - f1
 - roc_auc

Cross-validation strategy:
 - StratifiedKFold
 - n_splits = 5
 - shuffle = True
 - random_state = 42

Classifier setup complete.


In [8]:
# ============================================
# Cell 6: Run Baseline Classifier Comparison
# ============================================

print("Running baseline classifier comparison...\n")

# NOTE:
# This cell can take several minutes depending on hardware.
# Performance can be improved by:
# - Using a higher-memory environment (e.g., Colab High-RAM)
# - Running on a machine with more CPU cores
#
# This is due to parallel cross-validation (n_jobs=-1)
# and compute-intensive models such as SVM and MLP.

import time

baseline_results = []
total_models = len(candidate_classifiers)

for i, (name, clf) in enumerate(candidate_classifiers.items(), 1):
    print(f"[{i}/{total_models}] Evaluating: {name}")

    start_time = time.time()

    cv_results = cross_validate(
        estimator=clf,
        X=X_train,
        y=y_train_encoded,
        cv=cv_strategy,
        scoring=scoring_metrics,
        return_train_score=False,
        n_jobs=-1
    )

    elapsed = time.time() - start_time

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    result = {
        "classifier": name,
        "accuracy_mean": np.mean(cv_results["test_accuracy"]),
        "accuracy_std": np.std(cv_results["test_accuracy"]),
        "precision_mean": np.mean(cv_results["test_precision"]),
        "precision_std": np.std(cv_results["test_precision"]),
        "recall_mean": np.mean(cv_results["test_recall"]),
        "recall_std": np.std(cv_results["test_recall"]),
        "f1_mean": np.mean(cv_results["test_f1"]),
        "f1_std": np.std(cv_results["test_f1"]),
        "roc_auc_mean": np.mean(cv_results["test_roc_auc"]),
        "roc_auc_std": np.std(cv_results["test_roc_auc"]),
        "fit_time_sec": elapsed
    }

    baseline_results.append(result)

    print(f"Completed: {name} (time: {elapsed:.2f} sec)\n")

print("Baseline classifier comparison complete.")



Running baseline classifier comparison...

[1/9] Evaluating: Logistic Regression
Completed: Logistic Regression (time: 1.67 sec)

[2/9] Evaluating: Linear SVM
Completed: Linear SVM (time: 33.88 sec)

[3/9] Evaluating: RBF SVM
Completed: RBF SVM (time: 31.08 sec)

[4/9] Evaluating: Random Forest
Completed: Random Forest (time: 6.05 sec)

[5/9] Evaluating: Extra Trees
Completed: Extra Trees (time: 1.58 sec)

[6/9] Evaluating: Gradient Boosting
Completed: Gradient Boosting (time: 15.59 sec)

[7/9] Evaluating: KNN
Completed: KNN (time: 0.57 sec)

[8/9] Evaluating: Gaussian NB
Completed: Gaussian NB (time: 0.15 sec)

[9/9] Evaluating: MLP
Completed: MLP (time: 14.38 sec)

Baseline classifier comparison complete.


In [9]:
# ============================================
# Cell 7: Compile and Rank Results
# ============================================

print("Compiling and ranking baseline classifier results...\n")

# -------------------------------------------------
# Convert results to DataFrame
# -------------------------------------------------
df_baseline_results = pd.DataFrame(baseline_results)

# -------------------------------------------------
# Sort by primary metric (ROC-AUC)
# -------------------------------------------------
df_baseline_results = df_baseline_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Display results
# -------------------------------------------------
print("Baseline classifier comparison (sorted by ROC-AUC):\n")

try:
    display(df_baseline_results)
except:
    print(df_baseline_results)

# -------------------------------------------------
# Display top classifiers
# -------------------------------------------------
TOP_K = 3

print(f"\nTop {TOP_K} classifiers based on ROC-AUC:\n")

for i in range(min(TOP_K, len(df_baseline_results))):
    row = df_baseline_results.iloc[i]
    print(
        f"{i+1}. {row['classifier']} | "
        f"ROC-AUC: {row['roc_auc_mean']:.4f} | "
        f"F1: {row['f1_mean']:.4f} | "
        f"Accuracy: {row['accuracy_mean']:.4f}"
    )

print("\nRanking complete.")



Compiling and ranking baseline classifier results...

Baseline classifier comparison (sorted by ROC-AUC):



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,fit_time_sec
0,RBF SVM,0.777778,0.005835,0.773080,0.005561,0.786389,0.007962,0.779665,0.006092,0.861125,0.005117,31.078624
1,MLP,0.776042,0.006559,0.769598,0.014522,0.789167,0.025072,0.778821,0.008275,0.859080,0.008704,14.377259
2,Random Forest,0.766111,0.005287,0.771078,0.007182,0.757083,0.008361,0.763974,0.005326,0.847194,0.006478,6.046361
3,Extra Trees,0.756806,0.003456,0.756472,0.004165,0.757500,0.006202,0.756967,0.003734,0.842692,0.006798,1.579850
4,Gradient Boosting,0.750278,0.007898,0.747784,0.010058,0.755556,0.009986,0.751591,0.007489,0.830625,0.007015,15.587298
5,Linear SVM,0.727153,0.005853,0.712224,0.005775,0.762361,0.007289,0.736428,0.005756,0.799640,0.009323,33.884063
6,Logistic Regression,0.725833,0.004809,0.714878,0.004928,0.751389,0.009118,0.732647,0.005336,0.799256,0.009348,1.673223
7,KNN,0.727153,0.001957,0.718349,0.002187,0.747361,0.007849,0.732539,0.003258,0.793381,0.005663,0.570668
8,Gaussian NB,0.666667,0.005560,0.656834,0.006766,0.698333,0.011936,0.676869,0.005916,0.712219,0.006683,0.145323



Top 3 classifiers based on ROC-AUC:

1. RBF SVM | ROC-AUC: 0.8611 | F1: 0.7797 | Accuracy: 0.7778
2. MLP | ROC-AUC: 0.8591 | F1: 0.7788 | Accuracy: 0.7760
3. Random Forest | ROC-AUC: 0.8472 | F1: 0.7640 | Accuracy: 0.7661

Ranking complete.


In [10]:
# ============================================
# Cell 8: Tune Top Classifiers
# ============================================

print("Tuning top classifiers...\n")

import time
from itertools import product

tuned_results = []

# -------------------------------------------------
# Define parameter grids (small, controlled)
# -------------------------------------------------
param_grids = {
    "RBF SVM": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.1, 0.01]
    },

    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64, 32)],
        "alpha": [0.0001, 0.001],
        "learning_rate_init": [0.001, 0.0005]
    },

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20]
    }
}

# -------------------------------------------------
# Select top classifiers from baseline ranking
# -------------------------------------------------
TOP_K = 3
top_classifiers = [
    name for name in df_baseline_results["classifier"].head(TOP_K)
    if name in param_grids
]

print("Selected for tuning:")
for name in top_classifiers:
    print(f" - {name}")
print()

total_models = len(top_classifiers)

# -------------------------------------------------
# Tune each top classifier
# -------------------------------------------------
for i, name in enumerate(top_classifiers, 1):
    print(f"[{i}/{total_models}] Tuning: {name}")

    clf = candidate_classifiers[name]
    param_grid = param_grids[name]

    # Count parameter combinations
    n_combinations = len(list(product(*param_grid.values())))
    print(f"Parameter combinations: {n_combinations}")

    start_time = time.time()

    grid_search = GridSearchCV(
        estimator=clf,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv_strategy,
        verbose=1,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train_encoded)

    elapsed = time.time() - start_time

    best_model = grid_search.best_estimator_
    best_score = grid_search.best_score_

    print(f"Best ROC-AUC: {best_score:.4f}")
    print(f"Best params: {grid_search.best_params_}")
    print(f"Time: {elapsed:.2f} sec\n")

    tuned_results.append({
        "classifier": name,
        "best_roc_auc": best_score,
        "best_params": grid_search.best_params_,
        "fit_time_sec": elapsed
    })

print("Tuning complete.")



Tuning top classifiers...

Selected for tuning:
 - RBF SVM
 - MLP
 - Random Forest

[1/3] Tuning: RBF SVM
Parameter combinations: 9
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best ROC-AUC: 0.8790
Best params: {'C': 10, 'gamma': 'scale'}
Time: 269.68 sec

[2/3] Tuning: MLP
Parameter combinations: 8
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best ROC-AUC: 0.8669
Best params: {'alpha': 0.001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.0005}
Time: 136.09 sec

[3/3] Tuning: Random Forest
Parameter combinations: 6
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best ROC-AUC: 0.8488
Best params: {'max_depth': None, 'n_estimators': 200}
Time: 50.09 sec

Tuning complete.


In [13]:
# ============================================
# Cell 9: Save Results
# ============================================

print("Saving classifier comparison results...\n")

# -------------------------------------------------
# Convert tuned results to DataFrame
# -------------------------------------------------
df_tuned_results = pd.DataFrame(tuned_results)

if df_tuned_results.empty:
    raise ValueError("No tuned results found. Cell 8 may not have run successfully.")

# -------------------------------------------------
# Sort tuned results by best ROC-AUC
# -------------------------------------------------
df_tuned_results = df_tuned_results.sort_values(
    by="best_roc_auc",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Save baseline and tuned comparison tables
# -------------------------------------------------
df_baseline_results.to_csv(BASELINE_RESULTS_CSV, index=False)
df_tuned_results.to_csv(TUNED_RESULTS_CSV, index=False)

print(f"Saved baseline results: {BASELINE_RESULTS_CSV}")
print(f"Saved tuned results:    {TUNED_RESULTS_CSV}")

# -------------------------------------------------
# Save top two tuned classifiers to JSON
# -------------------------------------------------
TOP_K_FINAL = 2

top_models = df_tuned_results.head(TOP_K_FINAL)

top_classifiers_summary = []

for _, row in top_models.iterrows():
    top_classifiers_summary.append({
        "classifier": str(row["classifier"]),
        "roc_auc": float(row["best_roc_auc"]),
        "params": dict(row["best_params"])
    })

with open(BEST_CLASSIFIER_JSON, "w") as f:
    json.dump(top_classifiers_summary, f, indent=4)

print(f"Saved top {TOP_K_FINAL} classifier summaries: {BEST_CLASSIFIER_JSON}")

# -------------------------------------------------
# Display saved results
# -------------------------------------------------
print(f"\nTop {TOP_K_FINAL} tuned classifiers:")
for i, model in enumerate(top_classifiers_summary, 1):
    print(f"{i}. {model['classifier']}")
    print(f"   ROC-AUC: {model['roc_auc']:.4f}")
    print(f"   Params:  {model['params']}")

print("\nSave complete.")



Saving classifier comparison results...

Saved baseline results: /content/dip-ai-image-detection/models/classifier_comparison_baseline.csv
Saved tuned results:    /content/dip-ai-image-detection/models/classifier_comparison_tuned.csv
Saved top 2 classifier summaries: /content/dip-ai-image-detection/models/best_classifier_config.json

Top 2 tuned classifiers:
1. RBF SVM
   ROC-AUC: 0.8790
   Params:  {'C': 10, 'gamma': 'scale'}
2. MLP
   ROC-AUC: 0.8669
   Params:  {'alpha': 0.001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.0005}

Save complete.


In [14]:
# ============================================
# Cell 10: Summary and Recommendations
# ============================================

print("Summarizing classifier selection results...\n")

# -------------------------------------------------
# Safety checks
# -------------------------------------------------
if df_baseline_results.empty or df_tuned_results.empty:
    raise ValueError("Missing results. Ensure Cells 7 and 9 executed successfully.")

# -------------------------------------------------
# Identify best baseline classifier
# -------------------------------------------------
best_baseline_row = df_baseline_results.iloc[0]

print("Best baseline classifier:")
print(f"  Classifier: {best_baseline_row['classifier']}")
print(f"  ROC-AUC:    {best_baseline_row['roc_auc_mean']:.4f}")
print(f"  F1 Score:   {best_baseline_row['f1_mean']:.4f}")
print(f"  Accuracy:   {best_baseline_row['accuracy_mean']:.4f}")

# -------------------------------------------------
# Identify top tuned classifiers
# -------------------------------------------------
TOP_K_FINAL = 2
top_tuned_rows = df_tuned_results.head(TOP_K_FINAL)

print(f"\nTop {TOP_K_FINAL} tuned classifiers:")
for i, (_, row) in enumerate(top_tuned_rows.iterrows(), 1):
    print(f"  {i}. Classifier: {row['classifier']}")
    print(f"     ROC-AUC:    {row['best_roc_auc']:.4f}")
    print(f"     Parameters: {row['best_params']}")

# -------------------------------------------------
# Recommendation for subsequent notebook(s)
# -------------------------------------------------
recommended_classifiers = top_tuned_rows["classifier"].tolist()

print("\nRecommendation:")
print(
    "  The following classifiers are recommended for subsequent "
    "final training and independent test evaluation:"
)

for clf in recommended_classifiers:
    print(f"   - {clf}")

# -------------------------------------------------
# Interpretation
# -------------------------------------------------
print("\nInterpretation:")
print(
    "  Classifier selection was performed using the normalized "
    "25-feature DIP representation and stratified 5-fold cross-validation "
    "on the training set only. The held-out test set was not used in this "
    "notebook and remains reserved for final independent evaluation."
)

print(
    "\n  The baseline comparison identified the strongest classifier "
    "candidates, and light hyperparameter tuning was then applied to the "
    "top-performing models. The top two tuned classifiers are carried "
    "forward for final training and independent test evaluation in later notebooks."
)

print("\nClassifier selection notebook complete.")


Summarizing classifier selection results...

Best baseline classifier:
  Classifier: RBF SVM
  ROC-AUC:    0.8611
  F1 Score:   0.7797
  Accuracy:   0.7778

Top 2 tuned classifiers:
  1. Classifier: RBF SVM
     ROC-AUC:    0.8790
     Parameters: {'C': 10, 'gamma': 'scale'}
  2. Classifier: MLP
     ROC-AUC:    0.8669
     Parameters: {'alpha': 0.001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.0005}

Recommendation:
  The following classifiers are recommended for subsequent final training and independent test evaluation:
   - RBF SVM
   - MLP

Interpretation:
  Classifier selection was performed using the normalized 25-feature DIP representation and stratified 5-fold cross-validation on the training set only. The held-out test set was not used in this notebook and remains reserved for final independent evaluation.

  The baseline comparison identified the strongest classifier candidates, and light hyperparameter tuning was then applied to the top-performing models. The to